In [37]:
from linked_list import LinkedList

class Stack(LinkedList):
    
    def push(self, data):
        self.append(data)

    def peek(self):
        return self.tail.data

    def pop(self):
        ret = self.tail.data
        if self.length == 1:
            self.tail = self.head = None
        else:
            self.tail = self.tail.prev
            self.tail.next = None
        self.length -= 1
        return ret

Implementing the tokenized functions

In [38]:
def tokenize(expression):
    return expression.split()
expression = "12 2 4 + / 21 *"
elements = expression.split()

print(elements)

['12', '2', '4', '+', '/', '21', '*']


Processing an operator - in postfix evalution

The functions are all the same, the only thing that changes is the operator used to calculate the result variable.

It is very important to perform the operation between the elements that was second to to and the top elements. If we do it the other way around we'll get the wrong result.

For example, in the process_minus() function we do:

result = second_to_top - top # Correct
and not

result = top - second_to_top # Wrong

In [39]:
def process_minus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top - top
    stack.push(result)

def process_plus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top + top
    stack.push(result)

def process_times(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top * top
    stack.push(result)

def process_divide(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top / top
    stack.push(result)

def process_pow(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top ** top
    stack.push(result)

Evaluating postfix expressions
Here are the steps we need to follow to implement the evaluate_postfix() function.

Initialize an empty stack.
Tokenize the expression using the tokenize() funct
ion.
For each token  do:
If the token an opeator, call the corresponding function to proces

s it. For example, if we find a + we call the process_plus() fun
ction.
Otherwise (the token is a number) and we push that number to the top of the stack. Since each token is a string, we'll need to convert it to a float 
first.
Return the value that is left in the stack.

In [40]:
def evaluate_postfix(expression):
    tokens = tokenize(expression)
    stack = Stack()
    for token in tokens:
        if token == "+":
            process_plus(stack)
        elif token == "-":
            process_minus(stack)
        elif token == "*":
            process_times(stack)
        elif token == "/":
            process_divide(stack)
        elif token == "**":
            process_pow(stack)
        else:
            # The token is not an operator so it must be a number
            stack.push(float(token))
    return stack.pop()
    

In [41]:

expressions = [
    "4 6 -",
    "4 1 2 9 3 / * + 5 - *",
    "1 2 + 3 -",
    "1 2 - 3 +",
    "10 3 5 * 16 4 - / +",
    "5 3 4 2 - ** *",
    "12 2 4 + / 21 *",
    "1 1 + 2 **",
    "1 1 2 ** +"
]

for expression in expressions:
    print(evaluate_postfix(expression))

-2.0
8.0
0.0
2.0
11.25
45.0
42.0
4.0
2.0


Operator Precedece in Infix Notation : Implementing Shunting-Yard Algorithm

In infix operations - operation precedence rules defines the order in which we peform the operations. 

for example, 4+2*3 = 4+6 =10. times operator perform before the addition. 

If the operators have the same precedence, they are evaluated in the order they appear. 

In [42]:
precedence = {
    "+" : 1,
    "-" : 1,
    "*" : 2,
    "/" : 2,
    "**" : 3
}

print(precedence["+"] < precedence["*"])
print(precedence["+"] < precedence["-"])
print(precedence["/"] < precedence["**"])

True
False
True


Implementing Infix to Postfix function

###Processing tokens in infix to postfix conversions

Opening parentheses, " ( ":
Push the token into the stack. It will be used later when we find a closing parenthesis.

In [43]:
def process_opening_parenthesis(stack):
    stack.push("(")

Closing Parenthesis

While the top of stack isn't open parenthesis, "(", pop the top element, and append it to the postfix token list

2. Pop the opening parentheses out of the stack at the end

In [44]:
def process_closing_parenthesis(stack, postfix):
    #adding the tokens until we find open parenthesis
    while stack.peek() != "(":
        #stack.pop()
        postfix.append(stack.pop())
    stack.pop()

Operator evaluating :- 

Operator, +, -, *, / or **:
While the top of the stack is also an operator with a precedence greater than or equal to this operator, pop the top element and append it to the postfix token list.
Push the current operator to the top of the stack.

In [45]:
def process_operator(stack, postfix, operator):
    while len(stack) > 0 and stack.peek() in precedence and precedence[stack.peek()] >= precedence[operator]:
        postfix.append(stack.pop())
    stack.push(operator)

Numbers - 
Operand (any number):
Push the token into the the postfix token list.

In [46]:
def process_number(postfix, number):
    postfix.append(number)

In [47]:
def infix_to_postfix(expression):
    tokens = tokenize(expression)
    stack = Stack()
    postfix = []
    for token in tokens:
        if token == "(":
            process_opening_parenthesis(stack)
        elif token == ")":
            process_closing_parenthesis(stack, postfix)
        elif token in precedence:
            process_operator(stack, postfix, token)
        else:
            process_number(postfix, token)
    while len(stack) > 0:
        postfix.append(stack.pop())
    return " ".join(postfix)
            

 Evaluating Infix expressions - implementing evaluate() function that returns the value of an expression in infix notation

In [48]:
def evaluate(expression):
    postfix_expression = infix_to_postfix(expression)
    return evaluate_postfix(postfix_expression)

In [49]:
expressions = [
    "1 + 1",
    "1 * ( 2 - ( 1 + 1 ) )",
    "4 * ( 1 + 2 * ( 9 / 3 ) - 5 )",
    "10 + 3 * 5 / ( 16 - 4 * 1 )",
    "2 * 2 * 2 * 2 * 2 * 2 * 2 * 2",
    "2 ** 2 ** 2 ** 2 ** 2",
    "( 1 - 2 ) / ( 3 - 5 )",
    "9 / 8 * 8",
    "64 / ( 8 * 8 )",
]

for expression in expressions:
    print(evaluate(expression))

2.0
0.0
8.0
11.25
256.0
65536.0
0.5
9.0
1.0
